# Experiment 42 — Direct Forecaster Absolute-Performance Audit

This notebook addresses the next reviewer-facing issue:

> The paper reports relative downstream gains, but a reviewer may ask whether the **direct forecasters
> themselves are genuinely strong baselines** and what the corresponding absolute MSE/MAE values are.

No model training is performed here.

## Goals

For the frozen 96 downstream conditions (`6 datasets × 4 backbones × 4 horizons`), recover and report:

- Direct forecaster MSE
- Final historical-memory-integrated MSE
- Absolute MSE difference
- Relative MSE gain
- Direct MAE and Final MAE, if those fields were retained by the original experiments

The notebook produces:

1. a 96-condition absolute-metric CSV,
2. appendix-ready per-dataset/backbone/horizon tables,
3. dataset/backbone mean summaries,
4. a consistency audit verifying that the reported percentage gain matches the absolute MSE values,
5. an explicit missingness report for MAE.

## Interpretation rule

Absolute performance is reported to establish the quality and scale of the frozen direct forecasters.
We do **not** claim new SOTA from this table. The downstream study remains a controlled within-backbone
test of whether historical memory adds value beyond each direct forecaster.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 280)
pd.set_option("display.max_rows", 200)

DATASETS = [
    "Solar",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "ETTh1",
]

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
    "SegMoE",
]

HORIZONS = [96, 192, 336, 720]

ROOT_CANDIDATES = [
    Path("/data/dataset/strong_forecaster"),
    Path("/data/strong_forecaster"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find strong_forecaster root. Tried:\n"
        + "\n".join(f" - {p}" for p in ROOT_CANDIDATES)
    )

OUT_DIR = ROOT / "direct_forecaster_absolute_performance"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


ROOT: /data/dataset/strong_forecaster
OUT_DIR: /data/dataset/strong_forecaster/direct_forecaster_absolute_performance


## 1. Load the frozen Experiment 37 selected-condition table

In [2]:
preferred = (
    ROOT
    / "four_backbone_dataset_meta_analysis"
    / "condition_level_selected.csv"
)

if preferred.is_file():
    condition_csv = preferred
else:
    hits = sorted(ROOT.rglob("condition_level_selected.csv"))

    hits = [
        p for p in hits
        if "significance_metadata_recovery" not in str(p)
        and "direct_forecaster_absolute_performance" not in str(p)
    ]

    if not hits:
        raise FileNotFoundError(
            "Could not find condition_level_selected.csv. "
            "Run Experiment 37 v3 first."
        )

    condition_csv = hits[0]

print("Using:", condition_csv)

conditions = pd.read_csv(condition_csv)

required = {
    "Dataset",
    "Backbone",
    "Horizon",
    "MSEGain_pct",
}

missing = required - set(conditions.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

conditions = conditions[
    conditions["Dataset"].isin(DATASETS)
    & conditions["Backbone"].isin(BACKBONES)
    & conditions["Horizon"].astype(int).isin(HORIZONS)
].copy()

conditions["Horizon"] = conditions["Horizon"].astype(int)

KEY = ["Dataset", "Backbone", "Horizon"]

if len(conditions) != 96:
    raise RuntimeError(
        f"Expected 96 selected conditions; found {len(conditions)}."
    )

if conditions.duplicated(KEY).any():
    raise RuntimeError(
        "Duplicate selected conditions found."
    )

print("Columns available:")
for c in conditions.columns:
    print(" -", c)


Using: /data/dataset/strong_forecaster/four_backbone_dataset_meta_analysis/condition_level_selected.csv
Columns available:
 - Dataset
 - Backbone
 - Horizon
 - ProtocolFamily
 - ExperimentFamily
 - Direct_MSE
 - Final_MSE
 - MSEGain_pct
 - Direct_MAE
 - Final_MAE
 - MAEGain_pct
 - FinalMeanAlpha
 - OracleHeadroom_pct
 - ValidationGain_pct
 - CI_Low
 - CI_High
 - SignificantPositive
 - SignificantNegative
 - SourcePath
 - SourceScore
 - RecoveredFamilyBonus
 - FinalSourceScore


## 2. Detect absolute-metric columns

Historical experiment families used slightly different column names.
This cell maps them conservatively.


In [3]:
def find_col(df, candidates):
    lookup = {
        str(c).lower().replace("_", "").replace("-", ""): c
        for c in df.columns
    }

    for name in candidates:
        key = (
            str(name)
            .lower()
            .replace("_", "")
            .replace("-", "")
        )

        if key in lookup:
            return lookup[key]

    return None


ALIASES = {
    "DirectMSE": [
        "DirectMSE",
        "Direct_MSE",
        "BaselineMSE",
        "BaseMSE",
        "MSE_Direct",
        "DirectTestMSE",
    ],
    "FinalMSE": [
        "FinalMSE",
        "Final_MSE",
        "ShrinkAdaptiveMSE",
        "AdaptiveMSE",
        "OursMSE",
        "MSE_Final",
        "FinalTestMSE",
    ],
    "DirectMAE": [
        "DirectMAE",
        "Direct_MAE",
        "BaselineMAE",
        "BaseMAE",
        "MAE_Direct",
        "DirectTestMAE",
    ],
    "FinalMAE": [
        "FinalMAE",
        "Final_MAE",
        "ShrinkAdaptiveMAE",
        "AdaptiveMAE",
        "OursMAE",
        "MAE_Final",
        "FinalTestMAE",
    ],
}

detected = {
    canonical: find_col(
        conditions,
        candidates,
    )
    for canonical, candidates in ALIASES.items()
}

print("Detected metric columns:")
for k, v in detected.items():
    print(f" - {k}: {v}")


Detected metric columns:
 - DirectMSE: Direct_MSE
 - FinalMSE: Final_MSE
 - DirectMAE: Direct_MAE
 - FinalMAE: Final_MAE


## 3. Build a first-pass absolute-metric table from Experiment 37

If MSE fields are already present, no provenance search is needed for those conditions.
If they are missing, the next section searches historical CSV artifacts.


In [4]:
absolute = conditions[KEY + ["MSEGain_pct"]].copy()

for canonical, source_col in detected.items():
    if source_col is None:
        absolute[canonical] = np.nan
    else:
        absolute[canonical] = pd.to_numeric(
            conditions[source_col],
            errors="coerce",
        )

absolute["MSESource"] = np.where(
    absolute["DirectMSE"].notna()
    & absolute["FinalMSE"].notna(),
    "Experiment37Selected",
    "Missing",
)

absolute["MAESource"] = np.where(
    absolute["DirectMAE"].notna()
    & absolute["FinalMAE"].notna(),
    "Experiment37Selected",
    "Missing",
)

print(
    "MSE pairs already present:",
    int(
        (
            absolute["DirectMSE"].notna()
            & absolute["FinalMSE"].notna()
        ).sum()
    ),
    "/96",
)

print(
    "MAE pairs already present:",
    int(
        (
            absolute["DirectMAE"].notna()
            & absolute["FinalMAE"].notna()
        ).sum()
    ),
    "/96",
)

display(
    absolute.sort_values(KEY).head(24)
)


MSE pairs already present: 72 /96
MAE pairs already present: 72 /96


,Dataset,Backbone,Horizon,MSEGain_pct,DirectMSE,FinalMSE,DirectMAE,FinalMAE,MSESource,MAESource
0,ETTh1,PatchTST,96,-1.090730,0.378632,NaN,0.400303,NaN,Missing,Missing
1,ETTh1,PatchTST,192,-1.676780,0.413694,NaN,0.420978,NaN,Missing,Missing
2,ETTh1,PatchTST,336,-1.466894,0.441984,NaN,0.441434,NaN,Missing,Missing
3,ETTh1,PatchTST,720,-3.931421,0.461510,NaN,0.473867,NaN,Missing,Missing
4,ETTh1,SegMoE,96,-1.052852,0.429677,0.434201,0.437933,0.441780,Experiment37Selected,Experiment37Selected
5,ETTh1,SegMoE,192,-2.264470,0.481695,0.492603,0.473062,0.480069,Experiment37Selected,Experiment37Selected
6,ETTh1,SegMoE,336,-2.288568,0.535394,0.547647,0.505612,0.512787,Experiment37Selected,Experiment37Selected
7,ETTh1,SegMoE,720,-6.796243,0.657350,0.702025,0.571758,0.593895,Experiment37Selected,Experiment37Selected
8,ETTh1,TimeMixer,96,-0.567222,0.388839,NaN,0.402163,NaN,Missing,Missing
9,ETTh1,TimeMixer,192,-0.640128,0.442255,NaN,0.429601,NaN,Missing,Missing


## 4. Recover missing absolute metrics from historical CSV artifacts

This search is only a recovery pass. It excludes meta-analysis/recovery outputs to avoid circular provenance.

A historical row is eligible only if it contains an unambiguous dataset, backbone, horizon, and
Direct/Final metric pair.


In [5]:
def canon_dataset(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")

    return {
        "solar": "Solar",
        "solarenergy": "Solar",
        "weather": "Weather",
        "electricity": "Electricity",
        "ecl": "Electricity",
        "traffic": "Traffic",
        "exchange": "Exchange",
        "exchangerate": "Exchange",
        "etth1": "ETTh1",
    }.get(s, str(x).strip())


def canon_backbone(x):
    s = str(x).strip().lower().replace("-", "").replace("_", "")

    return {
        "patchtst": "PatchTST",
        "itransformer": "iTransformer",
        "timemixer": "TimeMixer",
        "segmoe": "SegMoE",
        "segmoeforecast": "SegMoE",
    }.get(s, str(x).strip())


def excluded_path(p):
    s = str(p)

    banned = [
        "four_backbone_dataset_meta_analysis/",
        "significance_metadata_recovery",
        "direct_forecaster_absolute_performance/",
    ]

    return any(token in s for token in banned)


historical_rows = []

for p in ROOT.rglob("*.csv"):
    if excluded_path(p):
        continue

    try:
        if p.stat().st_size > 500 * 1024 * 1024:
            continue
    except Exception:
        continue

    try:
        df = pd.read_csv(p)
    except Exception:
        continue

    c_dataset = find_col(
        df,
        ["Dataset", "Data"],
    )

    c_backbone = find_col(
        df,
        ["Backbone", "Model"],
    )

    c_horizon = find_col(
        df,
        ["Horizon", "PredLen", "pred_len", "H"],
    )

    c_direct_mse = find_col(
        df,
        ALIASES["DirectMSE"],
    )

    c_final_mse = find_col(
        df,
        ALIASES["FinalMSE"],
    )

    if None in {
        c_dataset,
        c_backbone,
        c_horizon,
        c_direct_mse,
        c_final_mse,
    }:
        continue

    c_direct_mae = find_col(
        df,
        ALIASES["DirectMAE"],
    )

    c_final_mae = find_col(
        df,
        ALIASES["FinalMAE"],
    )

    for idx, r in df.iterrows():
        dataset = canon_dataset(
            r[c_dataset]
        )

        backbone = canon_backbone(
            r[c_backbone]
        )

        try:
            horizon = int(
                r[c_horizon]
            )
        except Exception:
            continue

        if dataset not in DATASETS:
            continue

        if backbone not in BACKBONES:
            continue

        if horizon not in HORIZONS:
            continue

        dmse = pd.to_numeric(
            pd.Series([r[c_direct_mse]]),
            errors="coerce",
        ).iloc[0]

        fmse = pd.to_numeric(
            pd.Series([r[c_final_mse]]),
            errors="coerce",
        ).iloc[0]

        if pd.isna(dmse) or pd.isna(fmse):
            continue

        dmae = (
            pd.to_numeric(
                pd.Series([r[c_direct_mae]]),
                errors="coerce",
            ).iloc[0]
            if c_direct_mae is not None
            else np.nan
        )

        fmae = (
            pd.to_numeric(
                pd.Series([r[c_final_mae]]),
                errors="coerce",
            ).iloc[0]
            if c_final_mae is not None
            else np.nan
        )

        historical_rows.append({
            "Dataset": dataset,
            "Backbone": backbone,
            "Horizon": horizon,
            "DirectMSE_recovered": dmse,
            "FinalMSE_recovered": fmse,
            "DirectMAE_recovered": dmae,
            "FinalMAE_recovered": fmae,
            "RecoverySource": str(p),
            "RecoveryRow": int(idx),
        })

historical = pd.DataFrame(
    historical_rows,
    columns=[
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_recovered",
        "FinalMSE_recovered",
        "DirectMAE_recovered",
        "FinalMAE_recovered",
        "RecoverySource",
        "RecoveryRow",
    ],
)

print(
    "Historical candidate rows:",
    len(historical),
)

if len(historical):
    display(
        historical.sort_values(KEY).head(100)
    )


Historical candidate rows: 229


,Dataset,Backbone,Horizon,DirectMSE_recovered,FinalMSE_recovered,DirectMAE_recovered,FinalMAE_recovered,RecoverySource,RecoveryRow
37,ETTh1,PatchTST,96,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/three_backbone...,12
73,ETTh1,PatchTST,96,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/three_backbone...,12
121,ETTh1,PatchTST,96,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/cross_backbone...,8
145,ETTh1,PatchTST,96,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/cross_backbone...,8
40,ETTh1,PatchTST,192,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/three_backbone...,15
76,ETTh1,PatchTST,192,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/three_backbone...,15
123,ETTh1,PatchTST,192,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/cross_backbone...,10
147,ETTh1,PatchTST,192,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/cross_backbone...,10
43,ETTh1,PatchTST,336,0.441984,0.448468,0.441434,0.448939,/data/dataset/strong_forecaster/three_backbone...,18
79,ETTh1,PatchTST,336,0.441984,0.448468,0.441434,0.448939,/data/dataset/strong_forecaster/three_backbone...,18


## 5. Resolve historical duplicates by consistency with frozen MSEGain_pct

For each candidate metric pair,

\[
100\frac{\mathrm{DirectMSE}-\mathrm{FinalMSE}}
{\mathrm{DirectMSE}}
\]

must match the frozen Experiment 37 `MSEGain_pct`.

We therefore select the candidate with the smallest absolute gain mismatch.
This avoids relying on directory names alone.


In [6]:
gain_lookup = absolute.set_index(KEY)["MSEGain_pct"].to_dict()

resolved_rows = []

if len(historical):
    for key, g in historical.groupby(KEY):
        frozen_gain = gain_lookup.get(
            key,
            np.nan,
        )

        g = g.copy()

        g["RecoveredGain_pct"] = (
            (
                g["DirectMSE_recovered"]
                - g["FinalMSE_recovered"]
            )
            / g["DirectMSE_recovered"]
            * 100.0
        )

        g["GainMismatch"] = (
            g["RecoveredGain_pct"]
            - frozen_gain
        ).abs()

        g["PathLen"] = g["RecoverySource"].map(len)

        g = g.sort_values(
            [
                "GainMismatch",
                "PathLen",
                "RecoverySource",
            ]
        )

        resolved_rows.append(
            g.iloc[0].to_dict()
        )

recovered = pd.DataFrame(
    resolved_rows
)

print(
    "Resolved historical conditions:",
    len(recovered),
)

if len(recovered):
    display(
        recovered.sort_values(KEY).head(96)
    )


Resolved historical conditions: 72


,Dataset,Backbone,Horizon,DirectMSE_recovered,FinalMSE_recovered,DirectMAE_recovered,FinalMAE_recovered,RecoverySource,RecoveryRow,RecoveredGain_pct,GainMismatch,PathLen
0,ETTh1,PatchTST,96,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/cross_backbone...,8,-1.090730,0.000000e+00,100
1,ETTh1,PatchTST,192,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/cross_backbone...,10,-1.676780,0.000000e+00,100
2,ETTh1,PatchTST,336,0.441984,0.448468,0.441434,0.448939,/data/dataset/strong_forecaster/cross_backbone...,12,-1.466894,2.220446e-16,100
3,ETTh1,PatchTST,720,0.461510,0.479654,0.473867,0.489601,/data/dataset/strong_forecaster/cross_backbone...,14,-3.931421,0.000000e+00,100
4,ETTh1,TimeMixer,96,0.388839,0.391045,0.402163,0.404818,/data/dataset/strong_forecaster/three_backbone...,7,-0.567222,0.000000e+00,97
5,ETTh1,TimeMixer,192,0.442255,0.445086,0.429601,0.435802,/data/dataset/strong_forecaster/three_backbone...,17,-0.640128,0.000000e+00,100
6,ETTh1,TimeMixer,336,0.517918,0.525842,0.474665,0.484048,/data/dataset/strong_forecaster/three_backbone...,20,-1.529878,2.220446e-16,100
7,ETTh1,TimeMixer,720,0.512187,0.544449,0.484744,0.513554,/data/dataset/strong_forecaster/three_backbone...,23,-6.298939,8.881784e-16,100
8,ETTh1,iTransformer,96,0.392303,0.394281,0.407334,0.410792,/data/dataset/strong_forecaster/cross_backbone...,9,-0.504197,0.000000e+00,100
9,ETTh1,iTransformer,192,0.442639,0.449096,0.434737,0.442669,/data/dataset/strong_forecaster/cross_backbone...,11,-1.458748,0.000000e+00,100


## 6. Coalesce selected-table and recovered absolute metrics

In [7]:
if len(recovered):
    rec_cols = [
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_recovered",
        "FinalMSE_recovered",
        "DirectMAE_recovered",
        "FinalMAE_recovered",
        "RecoverySource",
        "RecoveredGain_pct",
        "GainMismatch",
    ]

    rec = recovered[rec_cols].copy()
else:
    rec = pd.DataFrame(
        columns=[
            "Dataset",
            "Backbone",
            "Horizon",
            "DirectMSE_recovered",
            "FinalMSE_recovered",
            "DirectMAE_recovered",
            "FinalMAE_recovered",
            "RecoverySource",
            "RecoveredGain_pct",
            "GainMismatch",
        ]
    )

full = absolute.merge(
    rec,
    on=KEY,
    how="left",
    validate="one_to_one",
)

full["DirectMSE_final"] = (
    full["DirectMSE"]
    .combine_first(
        full["DirectMSE_recovered"]
    )
)

full["FinalMSE_final"] = (
    full["FinalMSE"]
    .combine_first(
        full["FinalMSE_recovered"]
    )
)

full["DirectMAE_final"] = (
    full["DirectMAE"]
    .combine_first(
        full["DirectMAE_recovered"]
    )
)

full["FinalMAE_final"] = (
    full["FinalMAE"]
    .combine_first(
        full["FinalMAE_recovered"]
    )
)

full["MSEMetricSource"] = np.where(
    full["DirectMSE"].notna()
    & full["FinalMSE"].notna(),
    "Experiment37Selected",
    np.where(
        full["DirectMSE_final"].notna()
        & full["FinalMSE_final"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["MAEMetricSource"] = np.where(
    full["DirectMAE"].notna()
    & full["FinalMAE"].notna(),
    "Experiment37Selected",
    np.where(
        full["DirectMAE_final"].notna()
        & full["FinalMAE_final"].notna(),
        "HistoricalRecovery",
        "Missing",
    ),
)

full["AbsoluteMSEImprovement"] = (
    full["DirectMSE_final"]
    - full["FinalMSE_final"]
)

full["RecomputedMSEGain_pct"] = (
    full["AbsoluteMSEImprovement"]
    / full["DirectMSE_final"]
    * 100.0
)

full["GainConsistencyError"] = (
    full["RecomputedMSEGain_pct"]
    - full["MSEGain_pct"]
).abs()

display(
    full.sort_values(KEY)
)

full.to_csv(
    OUT_DIR / "direct_final_absolute_metrics_96_conditions.csv",
    index=False,
)


,Dataset,Backbone,Horizon,MSEGain_pct,DirectMSE,FinalMSE,DirectMAE,FinalMAE,MSESource,MAESource,DirectMSE_recovered,FinalMSE_recovered,DirectMAE_recovered,FinalMAE_recovered,RecoverySource,RecoveredGain_pct,GainMismatch,DirectMSE_final,FinalMSE_final,DirectMAE_final,FinalMAE_final,MSEMetricSource,MAEMetricSource,AbsoluteMSEImprovement,RecomputedMSEGain_pct,GainConsistencyError
0,ETTh1,PatchTST,96,-1.090730,0.378632,NaN,0.400303,NaN,Missing,Missing,0.378632,0.382762,0.400303,0.405032,/data/dataset/strong_forecaster/cross_backbone...,-1.090730,0.000000e+00,0.378632,0.382762,0.400303,0.405032,HistoricalRecovery,HistoricalRecovery,-0.004130,-1.090730,0.000000e+00
1,ETTh1,PatchTST,192,-1.676780,0.413694,NaN,0.420978,NaN,Missing,Missing,0.413694,0.420631,0.420978,0.429180,/data/dataset/strong_forecaster/cross_backbone...,-1.676780,0.000000e+00,0.413694,0.420631,0.420978,0.429180,HistoricalRecovery,HistoricalRecovery,-0.006937,-1.676780,0.000000e+00
2,ETTh1,PatchTST,336,-1.466894,0.441984,NaN,0.441434,NaN,Missing,Missing,0.441984,0.448468,0.441434,0.448939,/data/dataset/strong_forecaster/cross_backbone...,-1.466894,2.220446e-16,0.441984,0.448468,0.441434,0.448939,HistoricalRecovery,HistoricalRecovery,-0.006483,-1.466894,2.220446e-16
3,ETTh1,PatchTST,720,-3.931421,0.461510,NaN,0.473867,NaN,Missing,Missing,0.461510,0.479654,0.473867,0.489601,/data/dataset/strong_forecaster/cross_backbone...,-3.931421,0.000000e+00,0.461510,0.479654,0.473867,0.489601,HistoricalRecovery,HistoricalRecovery,-0.018144,-3.931421,0.000000e+00
4,ETTh1,SegMoE,96,-1.052852,0.429677,0.434201,0.437933,0.441780,Experiment37Selected,Experiment37Selected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.429677,0.434201,0.437933,0.441780,Experiment37Selected,Experiment37Selected,-0.004524,-1.052852,1.287859e-14
5,ETTh1,SegMoE,192,-2.264470,0.481695,0.492603,0.473062,0.480069,Experiment37Selected,Experiment37Selected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.481695,0.492603,0.473062,0.480069,Experiment37Selected,Experiment37Selected,-0.010908,-2.264470,1.154632e-14
6,ETTh1,SegMoE,336,-2.288568,0.535394,0.547647,0.505612,0.512787,Experiment37Selected,Experiment37Selected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.535394,0.547647,0.505612,0.512787,Experiment37Selected,Experiment37Selected,-0.012253,-2.288568,0.000000e+00
7,ETTh1,SegMoE,720,-6.796243,0.657350,0.702025,0.571758,0.593895,Experiment37Selected,Experiment37Selected,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.657350,0.702025,0.571758,0.593895,Experiment37Selected,Experiment37Selected,-0.044675,-6.796243,0.000000e+00
8,ETTh1,TimeMixer,96,-0.567222,0.388839,NaN,0.402163,NaN,Missing,Missing,0.388839,0.391045,0.402163,0.404818,/data/dataset/strong_forecaster/three_backbone...,-0.567222,0.000000e+00,0.388839,0.391045,0.402163,0.404818,HistoricalRecovery,HistoricalRecovery,-0.002206,-0.567222,0.000000e+00
9,ETTh1,TimeMixer,192,-0.640128,0.442255,NaN,0.429601,NaN,Missing,Missing,0.442255,0.445086,0.429601,0.435802,/data/dataset/strong_forecaster/three_backbone...,-0.640128,0.000000e+00,0.442255,0.445086,0.429601,0.435802,HistoricalRecovery,HistoricalRecovery,-0.002831,-0.640128,0.000000e+00


## 7. Consistency and coverage audit

The absolute MSE pair is accepted only if it reproduces the frozen percentage gain.
A small tolerance is allowed for CSV rounding.


In [8]:
MSE_TOL_PCT = 0.01

mse_complete = (
    full["DirectMSE_final"].notna()
    & full["FinalMSE_final"].notna()
)

mae_complete = (
    full["DirectMAE_final"].notna()
    & full["FinalMAE_final"].notna()
)

bad_gain = full[
    mse_complete
    & (
        full["GainConsistencyError"]
        > MSE_TOL_PCT
    )
].copy()

print(
    "Complete MSE pairs:",
    int(mse_complete.sum()),
    "/96",
)

print(
    "Complete MAE pairs:",
    int(mae_complete.sum()),
    "/96",
)

print(
    "MSE gain consistency failures:",
    len(bad_gain),
)

if len(bad_gain):
    display(
        bad_gain[
            KEY
            + [
                "MSEGain_pct",
                "DirectMSE_final",
                "FinalMSE_final",
                "RecomputedMSEGain_pct",
                "GainConsistencyError",
                "MSEMetricSource",
                "RecoverySource",
            ]
        ]
    )

    raise RuntimeError(
        "Absolute MSE recovery contains gain-inconsistent rows."
    )

print("PASS: all recovered MSE pairs are consistent with frozen gains.")


Complete MSE pairs: 96 /96
Complete MAE pairs: 96 /96
MSE gain consistency failures: 0
PASS: all recovered MSE pairs are consistent with frozen gains.


## 8. Dataset × backbone summary

This is the most compact appendix-facing summary.

Because raw MSE scales differ by dataset, **we do not average MSE across datasets**.
Within each dataset/backbone, we report the four horizons separately and an average only across horizons.


In [9]:
summary_rows = []

for dataset in DATASETS:
    for backbone in BACKBONES:
        g = full[
            (full["Dataset"] == dataset)
            & (full["Backbone"] == backbone)
        ].sort_values("Horizon")

        row = {
            "Dataset": dataset,
            "Backbone": backbone,
            "MeanDirectMSE_overH": (
                float(g["DirectMSE_final"].mean())
                if g["DirectMSE_final"].notna().any()
                else np.nan
            ),
            "MeanFinalMSE_overH": (
                float(g["FinalMSE_final"].mean())
                if g["FinalMSE_final"].notna().any()
                else np.nan
            ),
            "MeanMSEGain_pct_overH": float(
                g["MSEGain_pct"].mean()
            ),
            "Wins": int(
                (g["MSEGain_pct"] > 0).sum()
            ),
            "Ties": int(
                (g["MSEGain_pct"].abs() <= 1e-12).sum()
            ),
            "Losses": int(
                (g["MSEGain_pct"] < 0).sum()
            ),
        }

        if g["DirectMAE_final"].notna().all():
            row["MeanDirectMAE_overH"] = float(
                g["DirectMAE_final"].mean()
            )
            row["MeanFinalMAE_overH"] = float(
                g["FinalMAE_final"].mean()
            )
        else:
            row["MeanDirectMAE_overH"] = np.nan
            row["MeanFinalMAE_overH"] = np.nan

        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

display(summary)

summary.to_csv(
    OUT_DIR / "dataset_backbone_absolute_summary.csv",
    index=False,
)


,Dataset,Backbone,MeanDirectMSE_overH,MeanFinalMSE_overH,MeanMSEGain_pct_overH,Wins,Ties,Losses,MeanDirectMAE_overH,MeanFinalMAE_overH
0,Solar,PatchTST,0.196426,0.193343,1.547045,4,0,0,0.250307,0.248463
1,Solar,iTransformer,0.238350,0.209458,12.049133,4,0,0,0.261168,0.255080
2,Solar,TimeMixer,0.233234,0.220655,5.194140,4,0,0,0.271717,0.269121
3,Solar,SegMoE,0.269517,0.225702,13.179044,4,0,0,0.278548,0.268875
4,Weather,PatchTST,0.228065,0.223323,2.025585,4,0,0,0.263642,0.261528
5,Weather,iTransformer,0.259591,0.248788,4.204754,4,0,0,0.279783,0.276036
6,Weather,TimeMixer,0.244659,0.239735,1.814475,4,0,0,0.273641,0.272001
7,Weather,SegMoE,0.223747,0.218869,2.058731,4,0,0,0.256018,0.254421
8,Electricity,PatchTST,0.161653,0.161216,0.358499,3,0,1,0.254120,0.254527
9,Electricity,iTransformer,0.176814,0.175374,0.882490,4,0,0,0.267041,0.267463


## 9. Appendix-ready horizon table

This table retains the actual direct and final MSE for every downstream condition.
It is the strongest way to show that the historical-memory result is measured against a concrete,
frozen direct forecast rather than an abstract percentage baseline.


In [10]:
paper = full[
    [
        "Dataset",
        "Backbone",
        "Horizon",
        "DirectMSE_final",
        "FinalMSE_final",
        "MSEGain_pct",
        "DirectMAE_final",
        "FinalMAE_final",
        "MSEMetricSource",
        "MAEMetricSource",
    ]
].copy()

paper = paper.sort_values(
    ["Dataset", "Backbone", "Horizon"]
)

display(paper)

paper.to_csv(
    OUT_DIR / "paper_absolute_metrics_96_conditions.csv",
    index=False,
)

latex = paper.copy()

for c in [
    "DirectMSE_final",
    "FinalMSE_final",
    "DirectMAE_final",
    "FinalMAE_final",
]:
    latex[c] = latex[c].map(
        lambda v: (
            ""
            if pd.isna(v)
            else f"{v:.4f}"
        )
    )

latex["MSEGain_pct"] = latex["MSEGain_pct"].map(
    lambda v: f"{v:+.3f}"
)

(OUT_DIR / "paper_absolute_metrics_96_conditions.tex").write_text(
    latex.to_latex(
        index=False,
        escape=False,
    ),
    encoding="utf-8",
)

print("Saved paper-ready CSV and LaTeX table.")


,Dataset,Backbone,Horizon,DirectMSE_final,FinalMSE_final,MSEGain_pct,DirectMAE_final,FinalMAE_final,MSEMetricSource,MAEMetricSource
0,ETTh1,PatchTST,96,0.378632,0.382762,-1.090730,0.400303,0.405032,HistoricalRecovery,HistoricalRecovery
1,ETTh1,PatchTST,192,0.413694,0.420631,-1.676780,0.420978,0.429180,HistoricalRecovery,HistoricalRecovery
2,ETTh1,PatchTST,336,0.441984,0.448468,-1.466894,0.441434,0.448939,HistoricalRecovery,HistoricalRecovery
3,ETTh1,PatchTST,720,0.461510,0.479654,-3.931421,0.473867,0.489601,HistoricalRecovery,HistoricalRecovery
4,ETTh1,SegMoE,96,0.429677,0.434201,-1.052852,0.437933,0.441780,Experiment37Selected,Experiment37Selected
5,ETTh1,SegMoE,192,0.481695,0.492603,-2.264470,0.473062,0.480069,Experiment37Selected,Experiment37Selected
6,ETTh1,SegMoE,336,0.535394,0.547647,-2.288568,0.505612,0.512787,Experiment37Selected,Experiment37Selected
7,ETTh1,SegMoE,720,0.657350,0.702025,-6.796243,0.571758,0.593895,Experiment37Selected,Experiment37Selected
8,ETTh1,TimeMixer,96,0.388839,0.391045,-0.567222,0.402163,0.404818,HistoricalRecovery,HistoricalRecovery
9,ETTh1,TimeMixer,192,0.442255,0.445086,-0.640128,0.429601,0.435802,HistoricalRecovery,HistoricalRecovery


Saved paper-ready CSV and LaTeX table.


/tmp/ipykernel_111295/2673915562.py:48: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  latex.to_latex(


## 10. Reviewer-facing completion summary

Desired outcome:

- `Complete MSE pairs: 96/96`
- all gain consistency checks pass.

MAE may have lower coverage depending on which historical experiment family retained it.
That is acceptable: the paper can report full MSE coverage and explicitly state MAE coverage rather than
silently filling missing values.

If MSE reaches 96/96, the next manuscript revision should add:

1. a compact appendix table of Direct MSE → Final MSE,
2. one sentence in Section 6 clarifying that all percentage gains are computed against these frozen
   absolute direct-forecast errors,
3. a scope statement that these are controlled within-backbone comparisons, not a new cross-paper SOTA claim.


In [11]:
print("=" * 118)
print("EXPERIMENT 42 — DIRECT FORECASTER ABSOLUTE-PERFORMANCE AUDIT")
print("=" * 118)

print(
    f"Complete MSE pairs: "
    f"{int(mse_complete.sum())}/96"
)

print(
    f"Complete MAE pairs: "
    f"{int(mae_complete.sum())}/96"
)

print(
    "Maximum MSE-gain consistency error:",
    (
        float(
            full.loc[
                mse_complete,
                "GainConsistencyError",
            ].max()
        )
        if mse_complete.any()
        else np.nan
    ),
)

print("\nMSE source breakdown:")
print(
    full[
        "MSEMetricSource"
    ].value_counts(
        dropna=False
    ).to_string()
)

print("\nMAE source breakdown:")
print(
    full[
        "MAEMetricSource"
    ].value_counts(
        dropna=False
    ).to_string()
)

missing_mse = full[
    ~mse_complete
]

missing_mae = full[
    ~mae_complete
]

if len(missing_mse):
    print("\nMissing MSE conditions:")
    print(
        missing_mse[
            KEY + ["MSEGain_pct"]
        ]
        .sort_values(KEY)
        .to_string(index=False)
    )

if len(missing_mae):
    print(
        f"\nMAE missing for "
        f"{len(missing_mae)}/96 conditions."
    )

print("\nOutputs:")
for name in [
    "direct_final_absolute_metrics_96_conditions.csv",
    "dataset_backbone_absolute_summary.csv",
    "paper_absolute_metrics_96_conditions.csv",
    "paper_absolute_metrics_96_conditions.tex",
]:
    print(" -", OUT_DIR / name)


EXPERIMENT 42 — DIRECT FORECASTER ABSOLUTE-PERFORMANCE AUDIT
Complete MSE pairs: 96/96
Complete MAE pairs: 96/96
Maximum MSE-gain consistency error: 5.051514762044462e-14

MSE source breakdown:
Experiment37Selected    72
HistoricalRecovery      24

MAE source breakdown:
Experiment37Selected    72
HistoricalRecovery      24

Outputs:
 - /data/dataset/strong_forecaster/direct_forecaster_absolute_performance/direct_final_absolute_metrics_96_conditions.csv
 - /data/dataset/strong_forecaster/direct_forecaster_absolute_performance/dataset_backbone_absolute_summary.csv
 - /data/dataset/strong_forecaster/direct_forecaster_absolute_performance/paper_absolute_metrics_96_conditions.csv
 - /data/dataset/strong_forecaster/direct_forecaster_absolute_performance/paper_absolute_metrics_96_conditions.tex
